# 로지스틱 회귀

## 럭키백의 확률
- 데이터 준비하기

In [ ]:
import pandas as pd

fish = pd.read_csv('https://raw.githubusercontent.com/rickiepark/hongong-ml/master/fish.csv')
fish.head()

# 품종/무게/길이/대각선/높이/폭 => 6가지 정보
# 첫줄을 자동으로 컬럼명으로 가져옴 : 기본
# 인덱스는 자동으로 가져오지 않는다

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)>

In [ ]:
fish.info()

In [ ]:
# 품종에서 유일한 값을 추출  ==> 7개 종류

print( pd.unique(fish['Species']) )

In [5]:
# 데이터프레임에서 5개 열을 추출하여 넘파이로 변환하여 저장

# 입력 데이터 준비 : 5개 열의 숫자 데이터
fish_input = fish[ ['Weight','Length','Diagonal','Height','Width'] ].to_numpy()

In [ ]:
# 넘파이 이므로 : 행과 열의 값만  // 인덱스, 열제목 없음

print(fish_input[:5])

In [ ]:
# 타겟 데이터 준비 : 정답지 (품종)

fish_target = fish['Species'].to_numpy()
fish_target

In [8]:
# 훈련데이터(입력, 타겟), 테스트데이터(입력, 타겟)로 나눔

from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    fish_input, fish_target, random_state=42)

In [ ]:
# 표준화 전처리
# StandardScaler()
# -> 모든 피처들을 평균이 0, 분산이 1인 정규분포를 갖도록 만들어줌
# -> 즉, 표준화 해주는 방법

In [9]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()

# 훈련세트 : fit, transform
ss.fit(train_input)
train_scaled = ss.transform(train_input)

# 테스트세트 : transform
test_scaled = ss.transform(test_input)

# 필요한 데이터는 모두 준비 되었다.

## k-최근접 이웃 분류기의 확률 예측

In [ ]:
# 훈련 셋트로 훈련하고, 테스트 셋트로 점수 확인

from sklearn.neighbors import KNeighborsClassifier

# 최근접 이웃 갯수 3개  / 2,3,4,5 적용해서 스코어가 높은것 확인할 것
kn = KNeighborsClassifier(n_neighbors=3)

kn.fit(train_scaled, train_target)

print(kn.score(train_scaled, train_target))
print(kn.score(test_scaled, test_target))

In [ ]:
# 피쉬 데이터프레임에서 7가지 품종 :
# ['Bream' 'Roach' 'Whitefish' 'Parkki' 'Perch' 'Pike' 'Smelt']

# 이렇게 타깃데이터가 2개 이상이면 다중분류!
# 앞서는 1,0 : 이진분류
# 다중분류시 타킷을 숫자로 변경가능하나 사이킷런에서 문자열로 된 타킷값 사용가능

In [ ]:
print(kn.classes_)

# 출력 정렬됨

In [ ]:
# test_scaled

print(test_scaled[:5])


In [ ]:
# 테스트세트의 위 5개로 품종 예측

print(kn.predict(test_scaled[:5]))


In [ ]:
import numpy as np

# predict_proba 메소드: 클래스별 확률값 반환
proba = kn.predict_proba(test_scaled[:5])
print(np.round(proba, decimals=4))

In [ ]:
# 결과 설명
"""
 ['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']
------------------------------------------------------------------------------
 [  [0.     0.       1.      0.      0.      0.      0.    ]  => Perch
    [0.     0.       0.      0.      0.      1.      0.    ]  => Smelt
    [0.     0.       0.      1.      0.      0.      0.    ]  => Pike
    [0.     0.       0.6667  0.      0.3333  0.      0.    ]  => Perch or Roach
    [0.     0.       0.6667  0.      0.3333  0.      0.    ]] => Perch or Roach
"""

In [ ]:
# 네번째 샘플의 이웃 3개

distances, indexes = kn.kneighbors(test_scaled[3:4])  # 3 인덱스 모든컬럼 정보
print(train_target[indexes])

# 이웃 3개중에 Perch가 2개, Roach가 1개
# 따라서 Perch 확률 : 0.6667 // Roach 확률 : 0.3333

## 로지스틱 회귀
- 보통 회귀는 값을 예측하나, 로지스틱 회귀는 분류 모델이다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.arange(-5, 5, 0.1)    # -5 ~ 5 전까지의 0.1 간격으로

phi = 1 / (1 + np.exp(-z))   # 시그모이드 함수

plt.plot(z, phi)
plt.show()

# 시그모이드 함수 : 어떤값이든 -> 0 ~ 1 사이로 변환
# 로지스틱 회귀 : 0.5 까지 음성클래스 , 0.5 이상 양성클래스  => 분류이므로...

In [ ]:
z

### 로지스틱 회귀로 이진 분류 수행하기

In [ ]:
# 블리언 인덱싱 예제
# 블리언 인덱싱 : 배열에서 True 것만 출력

char_arr = np.array(['A', 'B', 'C', 'D', 'E'])
print( char_arr[ [True, False, True, False, False] ] )

In [ ]:
train_target

In [ ]:
# 이진분류 실습을 위해 2개 품종만 선택
# 훈련세트에서 Bream 또는 Smelt 만 True

bream_smelt_indexes = (train_target == 'Bream') | (train_target == 'Smelt')

bream_smelt_indexes

In [26]:
# 블리언 인덱싱
# 도미와 빙어 행만 선택

train_bream_smelt = train_scaled[bream_smelt_indexes]
target_bream_smelt = train_target[bream_smelt_indexes]


In [ ]:
from sklearn.linear_model import LogisticRegression

# 모델 생성 : 빈상태
lr = LogisticRegression()

# 모델 훈련
lr.fit(train_bream_smelt, target_bream_smelt)  # 도미와 빙어 훈련

In [ ]:
# 확률값 출력
print(lr.predict_proba(train_bream_smelt[:5]))

In [ ]:
# 위 결과 설명
'''
   Bream       Smelt
---------------------------------------
[[0.99759855 0.00240145]    ==> Bream
 [0.02735183 0.97264817]    ==> Smelt
 [0.99486072 0.00513928]    ==> Bream
 [0.98584202 0.01415798]    ==> Bream
 [0.99767269 0.00232731]]   ==> Bream
'''

In [ ]:
print(lr.classes_)

## 로지스틱 회귀로 다중 분류 수행하기

In [ ]:
#  LogisticRegression 클래스 -> 다중 분류 모델을 훈련
# C 속성 : 규제를 제어하는 매개변수
# --> C=20  // 기본 C=1 // 작은수록 규제가 커짐 // 완화하기 위해 C=20 지정
# max_iter : 최대반복횟수 // 기본 max_iter=100
# --> max_iter=100 설정시 횟수부족으로 경고 뜸 ==> Increase the number of iterations

lr = LogisticRegression(C=20, max_iter=1000)
lr.fit(train_scaled, train_target)

print(lr.score(train_scaled, train_target))
print(lr.score(test_scaled, test_target))


In [ ]:
print(lr.predict(test_scaled[:5]))

In [ ]:
lr.classes_

In [ ]:
# 각 클래스 확률값

proba = lr.predict_proba(test_scaled[:5])
print(np.round(proba, decimals=3))

In [ ]:
# 위 결과 설명
'''
['Bream', 'Parkki', 'Perch', 'Pike', 'Roach', 'Smelt', 'Whitefish']  ==> lr.classes_
------------------------------------------------------------------------------------
[[0.      0.014     0.841     0.      0.136   0.007     0.003]       ==> Perch
 [0.      0.003     0.044     0.      0.007   0.946     0.   ]       ==> Smelt
 [0.      0.        0.034     0.935   0.015   0.016     0.   ]       ==> Pike
 [0.011   0.034     0.306     0.007   0.567   0.        0.076]       ==> Roach
 [0.      0.        0.904     0.002   0.089   0.002     0.001]]      ==> perch
'''

In [ ]:
print(lr.classes_)

## 정리
### 예측 -> predict()
### 확률값 -> predict_proba()
